# 🎬 LTX-2 Video Generator — Free **Kaggle** UI

This is the same Gradio UI as the Colab notebook, but tuned for **free Kaggle Notebooks** which give you **20 GB on `/kaggle/working`** + **70 GB on `/tmp`** and a free P100 / T4 ×2 GPU (~30 hrs/week). Plenty of room for LTX-2 (~30 GB of weights).

## How to use this

1. Open <https://www.kaggle.com/code> → **+ New Notebook**.
2. **File → Import Notebook → From URL** → paste:
   `https://raw.githubusercontent.com/debzody/LTX-2/main/kaggle_ltx2_ui.ipynb`
3. Right sidebar:
   - **Settings → Accelerator → GPU T4 ×2** (or **GPU P100**).
   - **Settings → Persistence → Files only**.
   - **Settings → Internet → On** (required to download weights).
4. **Run All**.
5. The last cell prints a public `https://*.gradio.live` URL — open it.

## Why not Colab?
Free Colab's disk (~112 GB total, ~85 GB usable) is too tight for LTX-2's 22 GB checkpoint + 7 GB Gemma + Python deps.
Kaggle has more usable disk and the same free GPUs.

## 1. Sanity check

In [ ]:
!nvidia-smi || echo 'No GPU. Settings > Accelerator > GPU T4 x2'
!df -h / /kaggle/working /tmp 2>/dev/null

## 2. Clone the repo and pull the UI files

In [ ]:
import os
os.chdir('/kaggle/working')
if not os.path.exists('LTX-2'):
    !git clone --depth 1 https://github.com/Lightricks/LTX-2.git
%cd /kaggle/working/LTX-2
FORK = 'https://raw.githubusercontent.com/debzody/LTX-2/main'
!curl -sSL {FORK}/app.py -o app.py
!curl -sSL {FORK}/app_pipeline.py -o app_pipeline.py
!ls -la app.py app_pipeline.py

## 3. Install LTX-2 + Gradio (~5 min)

In [ ]:
!pip install -q uv
!uv pip install --system -e packages/ltx-core -e packages/ltx-pipelines
!uv pip install --system 'gradio>=4.44'

## 4. Hugging Face login (Gemma is gated)

Best on Kaggle: store the token as a **Kaggle Secret**.

1. Visit <https://huggingface.co/google/gemma-3-12b-it-qat-q4_0-unquantized> and click *Agree and access repository*.
2. Create a token (Read scope) at <https://huggingface.co/settings/tokens>.
3. In this notebook: **Add-ons → Secrets → Add new secret** → label `HF_TOKEN`, paste token.
4. Run the cell below.

In [ ]:
import os
from huggingface_hub import login, whoami

token = os.environ.get('HF_TOKEN')
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception as e:
        print('No Kaggle Secret named HF_TOKEN.', e)
if not token:
    from getpass import getpass
    token = getpass('HF token (hf_...): ').strip()
if not token:
    raise RuntimeError('No HF token. See cell 4 description above.')
login(token=token, add_to_git_credential=False)
os.environ['HF_TOKEN'] = token
print('Logged in as:', whoami()['name'])

## 5. Download model weights to `/kaggle/working/hf_cache` (~30 GB)

We download only the distilled checkpoint + Gemma — no upsampler — for the `one-stage` pipeline.

In [ ]:
import os, json, shutil
from huggingface_hub import hf_hub_download, snapshot_download
from huggingface_hub.utils import GatedRepoError

CACHE = '/kaggle/working/hf_cache'
os.environ['HF_HOME'] = CACHE
os.environ['HUGGINGFACE_HUB_CACHE'] = CACHE

free = shutil.disk_usage('/kaggle/working').free / (1024**3)
print(f'Free disk on /kaggle/working: {free:.1f} GB')
if free < 35:
    raise SystemExit(f'Need >=35 GB free, got {free:.1f} GB.')

print('Downloading distilled LTX-2 checkpoint (~22 GB)...')
ckpt = hf_hub_download(
    repo_id='Lightricks/LTX-2.3',
    filename='ltx-2.3-22b-distilled-1.1.safetensors',
    cache_dir=CACHE,
)
print('  ', ckpt)

print('Downloading Gemma text encoder (~7 GB)...')
try:
    gemma = snapshot_download(
        repo_id='google/gemma-3-12b-it-qat-q4_0-unquantized',
        cache_dir=CACHE,
        allow_patterns=['*.json', '*.safetensors', '*.model', 'tokenizer*', 'special_tokens*'],
    )
    print('  ', gemma)
except GatedRepoError as e:
    raise SystemExit(
        '\n\nGemma access denied. Accept the license at '
        'https://huggingface.co/google/gemma-3-12b-it-qat-q4_0-unquantized '
        'and use a Read token from the same account.\n\n' + str(e)
    )

with open('/kaggle/working/_paths.json', 'w') as f:
    json.dump({'ckpt': ckpt, 'gemma': gemma}, f)

!df -h /kaggle/working

## 6. Launch the UI with a public link

On the free T4 (16 GB VRAM) we run with `--quantization fp8-cast` and the UI's `one-stage` pipeline.

In [ ]:
import os, json
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
with open('/kaggle/working/_paths.json') as f:
    paths = json.load(f)
ckpt, gemma = paths['ckpt'], paths['gemma']
print('Checkpoint:', ckpt)
print('Gemma:', gemma)
!python app.py \
    --checkpoint-path "{ckpt}" \
    --gemma-root "{gemma}" \
    --quantization fp8-cast \
    --output-dir /kaggle/working/outputs \
    --host 0.0.0.0 --port 7860 \
    --share

Look for `Running on public URL: https://*.gradio.live` and open that.

### Tips for free Kaggle

- Default pipeline is **`one-stage`** — keep it on the free T4.
- Width/height ≤ 512, frames ≤ 65.
- Generated MP4s land in `/kaggle/working/outputs` — visible in the right-side **Output** panel.

## (Optional) 7. Stop the server

In [ ]:
!pkill -f 'python app.py' || true